This notebook collects and summarises the theoretical results used throughout the investigation: the estimators, their error properties, and the adaptations required for different payoffs. This is to be treated as a reference notebook when reading the implementations and experiments conducted.

# Reminder: Payoff definitions

Let $S_t$ denote the underlying price at time $t$, $K$ the strike, and $T$ the maturity. The options considered are:

| Contract | Payoff $H$ |
|:--|:--|
| European call | $H_{\mathrm{E}}=(S_T-K)^+$ |
| Arithmetic Asian call | $H_{\mathrm{A}}=\left(\frac{1}{M+1}\sum_{m=0}^{M}S_{t_m}-K\right)^+$ |
| Digital call | $H_{\mathrm{D}}=Q\mathbf{1}_{\{S_T>K\}}$ |
| Up-and-out call, discrete monitoring | $H_{\mathrm{UO}}^{(M)}=(S_T-K)^+\mathbf{1}_{\{\max_{0\leq m\leq M}S_{t_m}<B\}}$ |
| Up-and-out call, continuous monitoring | $H_{\mathrm{UO}}^{(c)}=(S_T-K)^+\mathbf{1}_{\{\sup_{0\leq t\leq T}S_t<B\}}$ |

For the Asian call, $0=t_0<t_1<\cdots<t_M=T$ are the averaging dates. For the discretely monitored barrier call, $0=t_0<t_1<\cdots<t_M=T$. Here $Q$ is the digital cash payment and $B$ is the upper barrier. Closed-form prices for continuously monitored barrier options are given by Rubinstein and Reiner (1991).

# 1. Finite Differences

Write the option value as

$$
V(S_0)=\mathbb{E}[Y(S_0,Z)],
$$

where $Y(S_0,Z)$ is the discounted payoff generated from the random input $Z$. From $N$ simulated paths,

$$
\widehat V_N(S_0)=\frac{1}{N}\sum_{i=1}^{N}Y(S_0,Z_i).
$$

Under risk-neutral valuation, an option price is the discounted expectation of its payoff (Merton, 1973). Finite Differences estimate $\Delta=V'(S_0)$ by perturbing $S_0$ and repricing. They act on the price function $V$, so the payoff itself need not be differentiable. This simulation-based construction is described by Glasserman (2004).

### Forward and Central Differences

The two estimators used are

$$
\widehat\Delta_{\mathrm{FD}}(N,h)
=\frac{\widehat V_N(S_0+h)-\widehat V_N(S_0)}{h},
$$

and

$$
\widehat\Delta_{\mathrm{CD}}(N,h)
=\frac{\widehat V_N(S_0+h)-\widehat V_N(S_0-h)}{2h}.
$$

Taylor expansion gives

$$
\frac{V(S_0+h)-V(S_0)}{h}
=\Delta+\frac{h}{2}V''(S_0)+O(h^2),
$$

whereas

$$
\frac{V(S_0+h)-V(S_0-h)}{2h}
=\Delta+\frac{h^2}{6}V'''(S_0)+O(h^4).
$$

Thus Forward Differences have bias $O(h)$, while the Central Difference bias is $O(h^2)$. If the price estimates at the bumped values are simulated independently, both variances are $O((Nh^2)^{-1})$; this finite-difference bias--variance trade-off is also derived by Haugh (2017, Section 1). The leading MSE surfaces are consequently

$$
\operatorname{MSE}_{\mathrm{FD}}(N,h)
\approx A_{\mathrm{FD}}h^2+\frac{B_{\mathrm{FD}}}{Nh^2},
$$

$$
\operatorname{MSE}_{\mathrm{CD}}(N,h)
\approx A_{\mathrm{CD}}h^4+\frac{B_{\mathrm{CD}}}{Nh^2}.
$$

Minimising these expressions gives

$$
h_{\mathrm{FD}}^*(N)
=\left(\frac{B_{\mathrm{FD}}}{A_{\mathrm{FD}}N}\right)^{1/4},
\qquad
h_{\mathrm{CD}}^*(N)
=\left(\frac{B_{\mathrm{CD}}}{2A_{\mathrm{CD}}N}\right)^{1/6}.
$$

### Common Random Numbers

Independent repricing wastes the natural similarity between two nearby options. Instead, the same shocks $Z_i$ can be used at each bumped value. This use of Common Random Numbers for simulated Greeks follows Glasserman (2004). For example,

$$
\widehat\Delta_{\mathrm{CD,CRN}}
=\frac{1}{N}\sum_{i=1}^{N}
\frac{Y(S_0+h,Z_i)-Y(S_0-h,Z_i)}{2h}.
$$

The variance of the numerator now contains

$$
-2\operatorname{Cov}\left(Y(S_0+h,Z),Y(S_0-h,Z)\right),
$$

The covariance is positive for nearby options, so this negative term reduces the variance. CRN does not change the finite-difference bias.

For a continuous, pathwise-differentiable payoff, the difference between paired observations is $O(h)$. Its variance is then $O(h^2)$, cancelling the $h^2$ in the denominator. This gives the approximations

$$
\operatorname{MSE}_{\mathrm{FD,CRN}}(N,h)
\approx A_{\mathrm{FD}}h^2+\frac{C_{\mathrm{FD}}}{N},
\qquad
\operatorname{MSE}_{\mathrm{CD,CRN}}(N,h)
\approx A_{\mathrm{CD}}h^4+\frac{C_{\mathrm{CD}}}{N}.
$$

Once the bias is small, reducing $h$ further gives little improvement. At extremely small values, floating-point cancellation eventually becomes relevant. At large $h$, bias dominates and the relative advantage of CRN fades.

# 2. Pathwise Differentiation

Finite Differences estimate Delta by perturbing $S_0$ and comparing two simulated prices. Pathwise Differentiation takes the opposite approach: we differentiate each simulated payoff with respect to $S_0$ and then average the resulting sensitivities. There is therefore no step size $h$ to choose. The general Pathwise estimator and its regularity conditions are developed by Broadie and Glasserman (1996).

Let

$$
V(S_0)=\mathbb{E}[Y(S_0,Z)],
$$

where $Y(S_0,Z)$ denotes one discounted payoff observation and $Z$ contains the random variables used to generate the path. Provided that $Y$ is differentiable with respect to $S_0$ almost surely and that differentiation can be moved inside the expectation,

$$
\Delta
=
\frac{\partial V}{\partial S_0}
=
\mathbb{E}\left[
\frac{\partial Y(S_0,Z)}{\partial S_0}
\right].
$$

Writing

$$
D
=
\frac{\partial Y(S_0,Z)}{\partial S_0},
$$

the Pathwise estimator is

$$
\widehat{\Delta}_{PW}
=
\frac{1}{N}\sum_{i=1}^N D^{(i)},
$$

where $N$ is the number of simulated paths. When the above interchange is valid, the estimator is unbiased and

$$
\operatorname{Var}(\widehat{\Delta}_{PW})
=
\frac{\operatorname{Var}(D)}{N}.
$$

As a side remark, this implies

$$
\operatorname{RMSE}(\widehat{\Delta}_{PW})
=
\frac{\sqrt{\operatorname{Var}(D)}}{\sqrt{N}}
=
O(N^{-1/2}).
$$

This is the usual Monte Carlo convergence rate. Unlike Finite Differences, there is no additional bias or variance coming from a step size.

### Implementation for the Different Payoffs

Under the Black--Scholes model (Black and Scholes, 1973),

$$
S_t
=
S_0
\exp\left(
\left(r-\frac{\sigma^2}{2}\right)t+\sigma W_t
\right),
$$

where $r$ is the risk-free rate, $\sigma$ is the volatility and $W_t$ is a standard Brownian motion. It follows that

$$
\frac{\partial S_t}{\partial S_0}
=
\frac{S_t}{S_0}.
$$

#### European Call

For the discounted payoff

$$
Y
=
e^{-rT}(S_T-K)^+,
$$

direct differentiation gives the pathwise observation

$$
D_{\mathrm{Eur}}
=
e^{-rT}
\mathbf{1}_{\{S_T>K\}}
\frac{S_T}{S_0}.
$$

Although the payoff is not differentiable at $S_T=K$, this does not affect the estimator because $\mathbb{P}(S_T=K)=0$. The estimator is therefore unbiased; the same European-call Pathwise derivation is presented by Haugh (2017).

#### Discretely Monitored Asian Call

Let

$$
A_M
=
\frac{1}{M+1}\sum_{m=0}^M S_{t_m}
$$

denote the arithmetic average over the $M+1$ dates, including $t_0=0$. Since every simulated price is proportional to $S_0$,

$$
\frac{\partial A_M}{\partial S_0}
=
\frac{1}{M+1}\sum_{m=0}^M\frac{S_{t_m}}{S_0}
=
\frac{A_M}{S_0}.
$$

The corresponding pathwise observation is

$$
D_{\mathrm{Asian}}
=
e^{-rT}
\mathbf{1}_{\{A_M>K\}}
\frac{A_M}{S_0}.
$$

As in the European case, the payoff kink causes no bias because the event $A_M=K$ has probability zero.

#### Digital Call: Smoothing

The direct method no longer works for a digital call, whose discounted payoff is

$$
Y
=
Qe^{-rT}\mathbf{1}_{\{S_T>K\}},
$$

where $Q$ is the fixed cash payment. For almost every simulated path, the indicator is locally constant with respect to $S_0$, so its pathwise derivative is zero. Averaging these derivatives would therefore give zero, even though the price of the option clearly depends on $S_0$. In this case, differentiation cannot be moved inside the expectation. This failure of the naïve Pathwise estimator for discontinuous payoffs, together with smoothing-based alternatives, is discussed by Liu and Hong (2011).

We therefore replace the indicator by the smooth approximation

$$
g_{\varepsilon_N}(S_T)
=
\Phi\left(
\frac{S_T-K}{\varepsilon_N}
\right),
\qquad
\varepsilon_N>0,
$$

where $\Phi$ is the standard normal cumulative distribution function. The parameter $\varepsilon_N$ controls how closely the smooth function approximates the original payoff. A smaller value gives a sharper transition around the strike, but also makes the estimator depend on a smaller number of paths close to $K$.

Since

$$
g_{\varepsilon_N}'(S_T)
=
\frac{1}{\varepsilon_N}
\phi\left(
\frac{S_T-K}{\varepsilon_N}
\right),
$$

where $\phi$ is the standard normal density, the smoothed estimator is

$$
\widehat{\Delta}_{PW,\varepsilon_N}
=
\frac{Qe^{-rT}}{N}
\sum_{i=1}^N
\frac{1}{\varepsilon_N}
\phi\left(
\frac{S_T^{(i)}-K}{\varepsilon_N}
\right)
\frac{S_T^{(i)}}{S_0}.
$$

This estimator is unbiased for the smoothed payoff, but biased for the original digital payoff. Under the usual regularity conditions,

$$
\operatorname{Bias}
=
O(\varepsilon_N^2),
\qquad
\operatorname{Var}
=
O\left(\frac{1}{N\varepsilon_N}\right).
$$

There is therefore a trade-off. Decreasing $\varepsilon_N$ reduces the smoothing bias, but increases the variance because fewer terminal prices contribute materially to the estimator and their weights grow like $1/\varepsilon_N$.

The MSE can be approximated by

$$
\operatorname{MSE}
\left(
\widehat{\Delta}_{PW,\varepsilon_N}
\right)
\approx
C_b\varepsilon_N^4
+
\frac{C_v}{N\varepsilon_N},
$$

where $C_b$ and $C_v$ are positive constants independent of $N$ and $\varepsilon_N$. Minimising with respect to $\varepsilon_N$ gives

$$
\varepsilon_{\mathrm{opt}}(N)
=
\left(
\frac{C_v}{4C_b}
\right)^{1/5}
N^{-1/5}.
$$

At this value,

$$
\operatorname{MSE}
=
O(N^{-4/5}),
\qquad
\operatorname{RMSE}
=
O(N^{-2/5}).
$$

The convergence is therefore slower than the usual $N^{-1/2}$ Pathwise rate. This is the cost of smoothing the payoff discontinuity.

#### Continuously Monitored Barrier Call: Brownian Bridge

The naive method also fails for barrier options. Direct differentiation treats the survival indicator as fixed for each simulated path. This captures the sensitivity of the terminal payoff, but ignores the fact that changing $S_0$ also changes the probability of hitting the barrier.

For a continuously monitored up-and-out call, we instead replace the binary barrier event by its Brownian-bridge conditional survival probability. Let

$$
t_m=m\Delta t,
\qquad
\Delta t=\frac{T}{M},
$$

and suppose that both endpoints $S_{t_m}$ and $S_{t_{m+1}}$ lie below the barrier $B$. Define

$$
a_m
=
\log\left(\frac{B}{S_{t_m}}\right).
$$

Conditional on the two endpoints, the probability of crossing the barrier during the interval $[t_m,t_{m+1}]$ is

$$
p_m
=
\exp\left(
-\frac{2a_ma_{m+1}}{\sigma^2\Delta t}
\right).
$$

The conditional probability that the path survives every interval is then

$$
P_{\mathrm{surv}}
=
\prod_{m=0}^{M-1}(1-p_m).
$$

This Brownian-bridge conditioning construction follows Glasserman and Staum (2001). If any simulated endpoint is at or above $B$, we set $P_{\mathrm{surv}}=0$. The conditional discounted payoff can now be written as

$$
Y_{BB}
=
e^{-rT}(S_T-K)^+P_{\mathrm{surv}}.
$$

The barrier indicator has been replaced by a probability that varies smoothly with the simulated path. We can therefore differentiate both the terminal payoff and the probability of surviving the barrier.

Since

$$
\frac{\partial a_m}{\partial S_0}
=
-\frac{1}{S_0},
$$

we obtain

$$
\frac{\partial p_m}{\partial S_0}
=
\frac{2p_m(a_m+a_{m+1})}
{S_0\sigma^2\Delta t},
$$

and hence

$$
\frac{\partial P_{\mathrm{surv}}}{\partial S_0}
=
-
P_{\mathrm{surv}}
\sum_{m=0}^{M-1}
\frac{1}{1-p_m}
\frac{\partial p_m}{\partial S_0}.
$$

The resulting pathwise observation is

$$
D_{BB}
=
e^{-rT}
\left[
\mathbf{1}_{\{S_T>K\}}
\frac{S_T}{S_0}
P_{\mathrm{surv}}
+
(S_T-K)^+
\frac{\partial P_{\mathrm{surv}}}{\partial S_0}
\right].
$$

The first term is the usual sensitivity of the terminal call payoff. The second accounts for the change in the probability of avoiding the barrier.

Under Black-Scholes, the Brownian-bridge probability is the exact conditional crossing probability. The conditioning therefore removes the payoff discontinuity without introducing smoothing bias. Provided that the pathwise observation has finite variance,

$$
\operatorname{RMSE}
\left(
\widehat{\Delta}_{PW,BB}
\right)
=
O(N^{-1/2}).
$$

Note that this construction applies to continuously monitored barriers. A discretely monitored barrier retains the discontinuous-indicator problem and would require a separate smoothing or conditioning argument.

### Relation with Finite Differences

Let $Y^*(S_0,Z)$ denote the payoff representation used by the Pathwise estimator: the original payoff for European and Asian calls, the smoothed payoff for the digital call, or the Brownian-bridge conditional payoff for the barrier call.

Using the same random inputs $Z^{(i)}$ at every perturbed value of $S_0$, the Forward- and Central-Difference estimators are

$$
\widehat{\Delta}_{FD}^{CRN}(h)
=
\frac{1}{N}
\sum_{i=1}^N
\frac{
Y^*(S_0+h,Z^{(i)})
-
Y^*(S_0,Z^{(i)})
}{h},
$$

and

$$
\widehat{\Delta}_{CD}^{CRN}(h)
=
\frac{1}{N}
\sum_{i=1}^N
\frac{
Y^*(S_0+h,Z^{(i)})
-
Y^*(S_0-h,Z^{(i)})
}{2h}.
$$

If $Y^*$ is differentiable at $S_0$ for almost every simulated path, then

$$
\widehat{\Delta}_{FD}^{CRN}(h)
\longrightarrow
\widehat{\Delta}_{PW},
\qquad
\widehat{\Delta}_{CD}^{CRN}(h)
\longrightarrow
\widehat{\Delta}_{PW}
\qquad
\text{as }h\to0.
$$

This limiting connection between finite-difference estimators with Common Random Numbers and Pathwise Differentiation is described by Glasserman (2004). Common Random Numbers are important here because they ensure that the difference between the two payoffs comes only from the perturbation of $S_0$. With independent simulations, the numerator also contains unrelated Monte Carlo noise, which becomes increasingly important as $h$ decreases.

For the raw digital and barrier payoffs, the Finite-Difference quotient may still converge path by path to the zero or incomplete naive derivative. This does not give the true Delta: it is precisely the problem caused by the payoff discontinuity. Smoothing or Brownian-bridge conditioning must therefore be applied before taking the pathwise limit.

# 3. Likelihood-Ratio

The Likelihood-Ratio method takes a different approach from both Finite Differences and Pathwise Differentiation. Instead of differentiating the payoff, it differentiates the probability density used to generate the simulated values. This makes the method applicable even when the payoff is discontinuous. The general method is developed by Broadie and Glasserman (1996).

Let $X$ denote the simulated terminal value or collection of path values, with density $f(x;S_0)$. The option price can be written as

$$
V(S_0)
=
\int H(x;S_0)f(x;S_0)\,dx,
$$

where $H(x;S_0)$ is the discounted payoff. Differentiating gives

$$
\Delta
=
\mathbb{E}\left[
\frac{\partial H(X;S_0)}{\partial S_0}
+
H(X;S_0)W(X)
\right],
$$

where

$$
W(X)
=
\frac{\partial}{\partial S_0}
\log f(X;S_0)
$$

is known as the score.

For the standard European, Digital and discretely monitored Barrier payoffs, $H$ does not depend explicitly on $S_0$ when the simulated states $X$ are held fixed. The estimator then reduces to

$$
\widehat{\Delta}_{LR}
=
\frac{1}{N}
\sum_{i=1}^N H^{(i)}W^{(i)}.
$$

Note that this is different from Pathwise Differentiation. Here, the simulated values are treated as fixed and the change in their probability density is measured instead.

Under the usual regularity conditions, the estimator is unbiased:

$$
\mathbb{E}[\widehat{\Delta}_{LR}]
=
\Delta.
$$

Its variance is

$$
\operatorname{Var}(\widehat{\Delta}_{LR})
=
\frac{\operatorname{Var}(HW)}{N},
$$

and hence

$$
\operatorname{RMSE}(\widehat{\Delta}_{LR})
=
O(N^{-1/2}).
$$

Although there is no step-size or smoothing bias, the variance can be large. This occurs when the payoff has a substantial approximately constant component, since multiplying this component by the zero-mean score adds noise without contributing to the expected Delta. The score also contains a factor proportional to $1/\sigma$, which can make the method unstable for low-volatility options.

### Payoff Centring

The score has mean zero:

$$
\mathbb{E}[W]
=
\int
\frac{\partial}{\partial S_0}
\log f(x;S_0)
f(x;S_0)\,dx
=
\frac{\partial}{\partial S_0}
\int f(x;S_0)\,dx
=
0.
$$

First consider the standard case in which $G=\partial H/\partial S_0=0$. We can then replace the payoff $H$ by $H-c$, where $c$ is any constant, without changing the expected value:

$$
\mathbb{E}[(H-c)W]
=
\mathbb{E}[HW]
-
c\mathbb{E}[W]
=
\mathbb{E}[HW].
$$

The centred estimator is

$$
\widehat{\Delta}_{LR,c}
=
\frac{1}{N}
\sum_{i=1}^N
(H^{(i)}-c)W^{(i)}.
$$

Payoff centring attempts to remove the part of the payoff that only contributes noise when multiplied by the score. This variance-reduction construction is presented by Glasserman (2004, Section 7.3). It is particularly useful for deep ITM options, where the payoff contains a large component that varies relatively little across paths.

Since the mean of the estimator does not depend on $c$, we choose $c$ to minimise

$$
\mathbb{E}\left[(H-c)^2W^2\right].
$$

Differentiating with respect to $c$ gives

$$
c^*
=
\frac{\mathbb{E}[HW^2]}
{\mathbb{E}[W^2]}.
$$

Thus, $c^*$ is not generally equal to the expected payoff. Paths with a large absolute score receive more weight because they contribute more strongly to the variance.

In practice, the optimal constant is estimated using a separate pilot sample:

$$
\widehat{c}^*
=
\frac{
\sum_{j=1}^{N_{\mathrm{pilot}}}
H^{(j)}(W^{(j)})^2
}{
\sum_{j=1}^{N_{\mathrm{pilot}}}
(W^{(j)})^2
}.
$$

Using a separate sample ensures that the estimated constant is independent of the observations used in the final Delta estimator. A more accurate pilot estimate should give a value closer to the theoretical variance-minimising constant, although it also introduces an additional computational cost.

When $G=\partial H/\partial S_0\neq0$, centring leaves this direct gradient term unchanged, so the one-path observation becomes $D_c=(H-c)W+G$. In this case, the variance-minimising constant is $c^*=\mathbb{E}[(HW+G)W]/\mathbb{E}[W^2]$, as required below for the Asian and continuously monitored barrier estimators.

### Implementation for the Different Payoffs

#### European Call

Under Black-Scholes,

$$
S_T
=
S_0
\exp\left(
\left(r-\frac{\sigma^2}{2}\right)T
+
\sigma\sqrt{T}Z
\right),
\qquad
Z\sim N(0,1).
$$

The score of the terminal lognormal density is

$$
W
=
\frac{Z}{S_0\sigma\sqrt{T}}.
$$

This European likelihood-ratio score is derived by Haugh (2017). For a European call,

$$
H
=
e^{-rT}(S_T-K)^+,
$$

so the estimator is

$$
\widehat{\Delta}_{LR}^{\mathrm{Eur}}
=
\frac{e^{-rT}}{N}
\sum_{i=1}^N
(S_T^{(i)}-K)^+
\frac{Z^{(i)}}{S_0\sigma\sqrt{T}}.
$$

For this score, the constant factor in $W^2$ cancels from the expression for $c^*$. The pilot estimate can therefore be written as

$$
\widehat{c}^*
=
\frac{\sum_{j=1}^{N_{\mathrm{pilot}}}H^{(j)}(Z^{(j)})^2}
{\sum_{j=1}^{N_{\mathrm{pilot}}}(Z^{(j)})^2}.
$$

#### Digital Call

For a digital call paying $Q$ at maturity,

$$
H
=
Qe^{-rT}\mathbf{1}_{\{S_T>K\}}.
$$

The same terminal score is used, giving

$$
\widehat{\Delta}_{LR}^{\mathrm{Dig}}
=
\frac{Qe^{-rT}}{N}
\sum_{i=1}^N
\mathbf{1}_{\{S_T^{(i)}>K\}}
\frac{Z^{(i)}}{S_0\sigma\sqrt{T}}.
$$

Unlike Pathwise Differentiation, no smoothing is required. The method differentiates the terminal density rather than the discontinuous indicator, so the ordinary estimator remains unbiased.

The optimal centring constant is again estimated through

$$
\widehat{c}^*
=
\frac{\sum_{j=1}^{N_{\mathrm{pilot}}}H^{(j)}(Z^{(j)})^2}
{\sum_{j=1}^{N_{\mathrm{pilot}}}(Z^{(j)})^2}.
$$

#### Discretely Monitored Asian Call

Let

$$
t_m=m\Delta t,
\qquad
\Delta t=\frac{T}{M},
$$

and let $Z_m$ denote the normal random variable used in the transition from $t_{m-1}$ to $t_m$. The joint density of the path can be factorised as

$$
f(S_{t_1},\ldots,S_{t_M};S_0)
=
f(S_{t_1};S_0)
\prod_{m=2}^M
f(S_{t_m}\mid S_{t_{m-1}}).
$$

When the simulated path values are held fixed, only the first transition depends explicitly on $S_0$. The score is therefore

$$
W
=
\frac{Z_1}{S_0\sigma\sqrt{\Delta t}}.
$$

This first-transition score follows the path likelihood-ratio construction in Haugh (2017, Section 3). In this investigation, however, the arithmetic average includes the initial value:

$$
A_M
=
\frac{1}{M+1}\sum_{m=0}^M S_{t_m},
\qquad
H
=
e^{-rT}(A_M-K)^+.
$$

Consequently, when the future simulated states are held fixed, the discounted payoff has the explicit derivative

$$
G_A
=
\frac{\partial H}{\partial S_0}
=
\frac{e^{-rT}}{M+1}
\mathbf{1}_{\{A_M>K\}}.
$$

The one-path observation and estimator are therefore

$$
D_A
=
HW+G_A,
\qquad
\widehat{\Delta}_{LR}^{\mathrm{Asian}}
=
\frac{1}{N}\sum_{i=1}^N D_A^{(i)}.
$$

Equivalently,

$$
\widehat{\Delta}_{LR}^{\mathrm{Asian}}
=
\frac{1}{N}\sum_{i=1}^N
\left[
e^{-rT}(A_M^{(i)}-K)^+
\frac{Z_1^{(i)}}{S_0\sigma\sqrt{\Delta t}}
+
\frac{e^{-rT}}{M+1}
\mathbf{1}_{\{A_M^{(i)}>K\}}
\right].
$$

Since

$$
\operatorname{Var}(W)
=
\frac{1}{S_0^2\sigma^2\Delta t}
=
\frac{M}{S_0^2\sigma^2T},
$$

the score becomes more variable as the number of monitoring dates increases. This explains why the MSE of the Asian LR estimator may increase with $M$, despite the estimator remaining unbiased.

Payoff centring leaves the direct gradient term unchanged:

$$
D_{A,c}
=
(H-c)W+G_A
=
D_A-cW.
$$

The optimal centring constant and its pilot estimate are therefore

$$
c_A^*
=
\frac{\mathbb{E}[D_AW]}{\mathbb{E}[W^2]},
\qquad
\widehat{c}_A^*
=
\frac{
\sum_{j=1}^{N_{\mathrm{pilot}}}D_A^{(j)}W^{(j)}
}{
\sum_{j=1}^{N_{\mathrm{pilot}}}(W^{(j)})^2
}.
$$

#### Discretely Monitored Barrier Call

The ordinary Likelihood-Ratio method also remains valid for a discretely monitored up-and-out call. Let

$$
H
=
e^{-rT}(S_T-K)^+
\mathbf{1}_{\left\{
\max_{1\leq m\leq M}S_{t_m}<B
\right\}},
$$

where $B$ is the barrier. Since the payoff is treated as a function of the fixed simulated states, its discontinuity does not prevent the density from being differentiated.

Using the first-transition score,

$$
\widehat{\Delta}_{LR}^{\mathrm{Bar,disc}}
=
\frac{1}{N}
\sum_{i=1}^N
H^{(i)}
\frac{Z_1^{(i)}}{S_0\sigma\sqrt{\Delta t}}.
$$

This is the naive LR implementation for a discretely monitored barrier. The continuous-monitoring case requires an additional term.

#### Continuously Monitored Barrier Call: Brownian Bridge

For continuous monitoring, we use the Brownian-bridge conditional survival probability

$$
P_{\mathrm{surv}}
=
\prod_{m=0}^{M-1}(1-p_m),
$$

where

$$
p_m
=
\exp\left(
-\frac{
2\log(B/S_{t_m})\log(B/S_{t_{m+1}})
}{
\sigma^2\Delta t
}
\right)
$$

is the conditional probability of crossing the barrier during interval $[t_m,t_{m+1}]$. If any simulated endpoint is at or above $B$, we set $P_{\mathrm{surv}}=0$.

The discounted conditional payoff is

$$
H_{BB}
=
e^{-rT}(S_T-K)^+P_{\mathrm{surv}}.
$$

Unlike the previous payoffs, $H_{BB}$ depends explicitly on $S_0$ through the first Brownian-bridge interval. The general LR formula must therefore be used:

$$
\Delta
=
\mathbb{E}\left[
H_{BB}W
+
G_{BB}
\right],
$$

where

$$
W
=
\frac{Z_1}{S_0\sigma\sqrt{\Delta t}}
$$

and

$$
G_{BB}
=
e^{-rT}(S_T-K)^+
\frac{\partial P_{\mathrm{surv}}}{\partial S_0}.
$$

When the remaining simulated states are held fixed, only the first crossing probability depends explicitly on $S_0$. Writing

$$
p_0
=
\exp\left(
-\frac{
2\log(B/S_0)\log(B/S_{t_1})
}{
\sigma^2\Delta t
}
\right),
$$

we obtain

$$
\frac{\partial P_{\mathrm{surv}}}{\partial S_0}
=
-
\frac{
2p_0\log(B/S_{t_1})
}{
S_0\sigma^2\Delta t
}
\prod_{m=1}^{M-1}(1-p_m).
$$

The estimator is therefore

$$
\widehat{\Delta}_{LR}^{BB}
=
\frac{1}{N}
\sum_{i=1}^N
e^{-rT}(S_T^{(i)}-K)^+
\left[
P_{\mathrm{surv}}^{(i)}
\frac{Z_1^{(i)}}{S_0\sigma\sqrt{\Delta t}}
+
\frac{\partial P_{\mathrm{surv}}^{(i)}}{\partial S_0}
\right].
$$

Only the first transition appears in the LR score, and only the first Brownian-bridge probability is differentiated explicitly. The effect of $S_0$ on the remaining simulated states and bridge intervals is already accounted for through the score.

Payoff centring can also be applied here. Define the uncentred one-path observation

$$
D_{BB}
=
H_{BB}W+G_{BB}.
$$

The centred observation is

$$
D_{BB,c}
=
(H_{BB}-c)W+G_{BB}
=
D_{BB}-cW.
$$

Since the estimator now contains the additional gradient term $G_{BB}$, the optimal constant is

$$
c_{BB}^*
=
\frac{
\mathbb{E}[D_{BB}W]
}{
\mathbb{E}[W^2]
}
=
\frac{
\mathbb{E}[H_{BB}W^2+G_{BB}W]
}{
\mathbb{E}[W^2]
}.
$$

Using a separate pilot sample, this is estimated by

$$
\widehat{c}_{BB}^*
=
\frac{
\sum_{j=1}^{N_{\mathrm{pilot}}}
D_{BB}^{(j)}W^{(j)}
}{
\sum_{j=1}^{N_{\mathrm{pilot}}}
(W^{(j)})^2
}.
$$

The additional $G_{BB}W$ term is required because the Brownian-bridge survival probability depends explicitly on $S_0$. Using the standard formula $\mathbb{E}[H_{BB}W^2]/\mathbb{E}[W^2]$ would not give the variance-minimising constant for the full barrier estimator in general.

# References

- Black, F. and Scholes, M. (1973). “The Pricing of Options and Corporate Liabilities.” *Journal of Political Economy*, 81(3), 637--654. https://doi.org/10.1086/260062

- Broadie, M. and Glasserman, P. (1996). “Estimating Security Price Derivatives Using Simulation.” *Management Science*, 42(2), 269--285. https://doi.org/10.1287/mnsc.42.2.269

- Glasserman, P. (2004). *Monte Carlo Methods in Financial Engineering*. New York: Springer. https://doi.org/10.1007/978-0-387-21617-1

- Glasserman, P. and Staum, J. (2001). “Conditioning on One-Step Survival for Barrier Option Simulations.” *Operations Research*, 49(6), 923--937. https://doi.org/10.1287/opre.49.6.923.10018

- Haugh, M. (2017). *Estimating the Greeks*. IEOR E4703 lecture notes, Columbia University. https://www.columbia.edu/~mh2078/MonteCarlo/MCS_Greeks.pdf

- Liu, G. and Hong, L. J. (2011). “Kernel Estimation of the Greeks for Options with Discontinuous Payoffs.” *Operations Research*, 59(1), 96--108. https://doi.org/10.1287/opre.1100.0844

- Merton, R. C. (1973). “Theory of Rational Option Pricing.” *The Bell Journal of Economics and Management Science*, 4(1), 141--183. https://doi.org/10.2307/3003143

- Rubinstein, M. and Reiner, E. (1991). “Breaking Down the Barriers.” *Risk*, 4(8), 28--35.